# 🚀 Zero-Cost KDPA/CBK Compliance Fine-Tuning
This notebook demonstrates how to fine-tune Llama-3-8B on KDPA/CBK compliance data using QLoRA. This is a zero-investment pipeline designed for Google Colab Free Tier (T4 GPU).

**Note for IDE Users:** If you see "Module not found" warnings for `torch`, `transformers`, or `google.colab`, please ignore them in your local editor. These modules are pre-installed or installed via the cell below when running in the intended Google Colab environment.

In [ ]:
%pip install -q -U transformers peft bitsandbytes trl datasets accelerate

In [ ]:
from google.colab import drive  # type: ignoredrive.mount('/content/drive')

In [ ]:
import torch  # type: ignorefrom transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments  # type: ignorefrom peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training  # type: ignorefrom trl import SFTTrainer  # type: ignorefrom datasets import load_dataset  # type: ignore
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
dataset_path = "/content/drive/MyDrive/Costloci/kdpa_finetune.jsonl"

# 1. Load Quantized Model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

# 2. LoRA Config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)

# 3. Load Dataset
dataset = load_dataset("json", data_files=dataset_path, split="train")

def formatting_func(example):
    return f"### Instruction: {example['instruction']}\n### Input: {example['input']}\n### Output: {example['output']}"

# 4. Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    max_seq_length=1024,
    tokenizer=AutoTokenizer.from_pretrained(model_id),
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=2,
        max_steps=50, # Adjust based on dataset size
        learning_rate=2e-4,
        fp16=True,
        logging_steps=1,
        output_dir="outputs",
        optim="paged_adamw_8bit"
    ),
    formatting_func=formatting_func
)

trainer.train()

# 5. Save
model.save_pretrained("/content/drive/MyDrive/Costloci/adapters")